# Data complexity effects on synthetic data quality

## Step 1: Generate source synthetic data

This notebook uses _make_classification()_ to produce synthetic data to be used as input to fit data synthetizers.


## Load libraries

In [ ]:
!pip install sdv

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from torch import manual_seed
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sdv.metadata import Metadata


## Experiment parameters

In [ ]:
EXP = "syndaite"

PROJECT_DIR = Path.cwd().resolve()

def project_path(path):
  path = Path(path)
  return path if path.is_absolute() else PROJECT_DIR / path


def relative_to_project(path):
  return str(project_path(path).resolve().relative_to(PROJECT_DIR))


INPUT_DIR = PROJECT_DIR / "data" / "synth_input" / EXP
RESULT_DIR = PROJECT_DIR / "results" / EXP
FIG_DIR = PROJECT_DIR / "results" / "figs" / EXP

RANDOM_SEEDS = [42, 123, 1234]
TEST_SIZE = 0.33
TARGET_NAME = "target"


In [ ]:
# Set seeds
np.random.seed(RANDOM_SEEDS[0])
manual_seed(RANDOM_SEEDS[0])
random.seed(RANDOM_SEEDS[0])

In [ ]:
def get_feature_cols(df, target_col):
  return [c for c in df.columns if c != target_col]

def get_metadata(df, feature_cols, target_col):
  meta = Metadata.detect_from_dataframe(df)

  for c in feature_cols:
    meta.update_column(column_name=c, sdtype="numerical")
  meta.update_column(column_name=target_col, sdtype="categorical")
  meta.validate()
  return(meta)


In [ ]:
from itertools import product

def my_product(inp):
    return (dict(zip(inp.keys(), values)) for values in product(*inp.values()))

params = {'random_state' : RANDOM_SEEDS,
          'n_samples' : [500, 1000, 2000], 
          'class_sep' : [0.5, 1., 2., 4.],
          'n_features': [10, 20],
          'n_informative': [2, 4],
          'n_redundant': [2],
          'n_repeated': [0],
          'n_classes': [2],
          'n_clusters_per_class' : [1, 2],
          'flip_y' : [0, 0.01, 0.1, 0.2] 
          }

all_params = list(my_product(params))
len(all_params)

## Generate and save source synthetic data

In [ ]:
for path in [INPUT_DIR, FIG_DIR]:
  path.mkdir(parents=True, exist_ok=True)

manifest_rows = []

i = 1
for param in all_params:

    if(i % 100 == 0):
        print("Generating dataset " + str(i) + " of " + str(len(all_params)) + "...")

    dataset_name = "D" + str(i)
    
    X, y = make_classification(**param)
    column_names = ['attr' + str(j) for j in np.arange(1, X.shape[1] + 1)]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=param["random_state"]
    )

    df = pd.DataFrame(X, columns=column_names)
    df[TARGET_NAME] = pd.Series(y, dtype="category")

    train = pd.DataFrame(X_train, columns=column_names)
    train[TARGET_NAME] = pd.Series(y_train, dtype="category")

    test = pd.DataFrame(X_test, columns=column_names)
    test[TARGET_NAME] = pd.Series(y_test, dtype="category")

    meta = get_metadata(train, column_names, target_col=TARGET_NAME)

    full_path = INPUT_DIR / f"{EXP}_{dataset_name}_full.csv"
    train_path = INPUT_DIR / f"{EXP}_{dataset_name}_train.csv"
    test_path = INPUT_DIR / f"{EXP}_{dataset_name}_test.csv"
    metadata_path = INPUT_DIR / f"{EXP}_{dataset_name}_metadata.json"

    df.to_csv(full_path, index=False)
    train.to_csv(train_path, index=False)
    test.to_csv(test_path, index=False)
    meta.save_to_json(str(metadata_path))

    manifest_row = {
        "Dataset": dataset_name,
        "base_path" : str(PROJECT_DIR),
        "full_path": relative_to_project(full_path),
        "train_path": relative_to_project(train_path),
        "test_path": relative_to_project(test_path),
        "metadata_path": relative_to_project(metadata_path),
        "target_col": TARGET_NAME,
    }
    manifest_row.update(param)
    manifest_rows.append(manifest_row)

    i += 1

manifest = pd.DataFrame(manifest_rows)

manifest.to_csv(INPUT_DIR / f"{EXP}_manifest.csv", index=False)
manifest


## Quick checks

In [ ]:
for row in manifest.itertuples(index=False):
    train = pd.read_csv(project_path(row.train_path))
    print(row.Dataset)
    print(train[TARGET_NAME].value_counts().sort_index())
    print()
